In [1]:
# import modules
import numpy as np
import tensorflow as tf
from tensorflow import keras
import pandas as pd
import seaborn as sns
from pylab import rcParams
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupShuffleSplit
from scipy import stats
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from tensorflow.keras import optimizers
from datetime import datetime
from kerastuner.tuners import RandomSearch
import tensorflow_addons as tfa
from keras.utils import to_categorical 
from keras.regularizers import l2
import os



r_seed = 42

WORKDIR = 'C:/Users/thewr/git/LSTM_giroscopio_acelerometro.git/'
np.random.seed(r_seed)
tf.random.set_seed(r_seed)
data_hora= datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
LOG_DIR = WORKDIR + "/Code/DataPrep/"+data_hora

C:\Users\thewr\AppData\Local\Temp\ipykernel_41756\2565918290.py:15: DeprecationWarning: `import kerastuner` is deprecated, please use `import keras_tuner`.
  from kerastuner.tuners import RandomSearch
c:\Users\thewr\git\LSTM_giroscopio_acelerometro.git\.venv\lib\site-packages\tensorflow_addons\utils\tfa_eol_msg.py:23: UserWarning: 

TensorFlow Addons (TFA) has ended development and introduction of new features.
TFA has entered a minimal maintenance and release mode until a planned end of life in May 2024.
Please modify downstream libraries to take dependencies from other repositories in our TensorFlow community (e.g. Keras, Keras-CV, and Keras-NLP). 

For more information see: https://github.com/tensorflow/addons/issues/2807 

  warnings.warn(


In [2]:
date = datetime.now()
date = date.strftime("%d/%m/%Y_%H:%M")
arq = datetime.now()
arq = arq.strftime("%d_%m_%Y_%H_%M")

In [3]:
def combine_csv_data(directory):
    # Specify the directory containing the TXT files

    # Create an empty list to store the DataFrames
    dfs = []

    # Iterate through the TXT files in the directory
    for filename in os.listdir(directory):
        if filename.endswith(".txt"):
            # Read the CSV data from the TXT file, assuming no header row
            df = pd.read_csv(os.path.join(directory, filename), header=None)

            # Append the DataFrame to the list
            dfs.append(df)

    # Concatenate the DataFrames into a single DataFrame
    combined_df = pd.concat(dfs, ignore_index=True)

    # Print the combined DataFrame
    return combined_df

In [4]:
## get dataset

# select only sensor data from smart watches
csv_phone_gyro_dir = WORKDIR +"Data/Raw/phone/gyro"
csv_phone_accel_dir = WORKDIR +"Data/Raw/phone/accel"
csv_watch_gyro_dir = WORKDIR +"Data/Raw/watch/gyro"
csv_watch_accel_dir = WORKDIR +"Data/Raw/watch/accel"

In [5]:
df_csv_phone_gyro = combine_csv_data(csv_phone_gyro_dir)

In [6]:
df_csv_phone_gyro.columns = ['pessoa_gyro','atividade_gyro','tempo_gyro','giroscópio_phone_x','giroscópio_phone_Y','giroscópio_phone_Z']

In [7]:
df_csv_phone_gyro.head()

,pessoa_gyro,atividade_gyro,tempo_gyro,giroscópio_phone_x,giroscópio_phone_Y,giroscópio_phone_Z
0,1600,A,252207918580802,-0.853210,0.297226,0.8901825;
1,1600,A,252207968934806,-0.875137,0.015472,0.16223145;
2,1600,A,252208019288809,-0.720169,0.388489,-0.28401184;
3,1600,A,252208069642813,-0.571640,1.227402,-0.2416687;
4,1600,A,252208119996817,-0.380493,1.202835,-0.21313477;


In [8]:
df_csv_phone_gyro.shape

(3608635, 6)

In [9]:
df_csv_phone_gyro.set_index(["pessoa_gyro","atividade_gyro","tempo_gyro"],inplace=True)

In [10]:
df_csv_phone_gyro.head()

giroscópio_phone_x  \
pessoa_gyro atividade_gyro tempo_gyro                            
1600        A              252207918580802           -0.853210   
                           252207968934806           -0.875137   
                           252208019288809           -0.720169   
                           252208069642813           -0.571640   
                           252208119996817           -0.380493   

                                            giroscópio_phone_Y  \
pessoa_gyro atividade_gyro tempo_gyro                            
1600        A              252207918580802            0.297226   
                           252207968934806            0.015472   
                           252208019288809            0.388489   
                           252208069642813            1.227402   
                           252208119996817            1.202835   

                                           giroscópio_phone_Z  
pessoa_gyro atividade_gyro tempo_gyro                          
1600        A              252207918580802         0.8901825;  
                           252207968934806        0.16223145;  
                           252208019288809       -0.28401184;  
                           252208069642813        -0.2416687;  
                           252208119996817       -0.21313477;

In [11]:
df_csv_phone_accel = combine_csv_data(csv_phone_accel_dir)

In [12]:
df_csv_phone_accel.shape

(4804403, 6)

In [13]:
df_csv_phone_accel.columns = ['pessoa_acelelômetro','atividade_acelerômetro','tempo_acelelômetro','acelelômetro_phone_x','acelelômetro_phone_Y','acelelômetro_phone_Z']

In [14]:
df_csv_phone_accel.head()

,pessoa_acelelômetro,atividade_acelerômetro,tempo_acelelômetro,acelelômetro_phone_x,acelelômetro_phone_Y,acelelômetro_phone_Z
0,1600,A,252207666810782,-0.364761,8.793503,1.0550842;
1,1600,A,252207717164786,-0.879730,9.768784,1.0169983;
2,1600,A,252207767518790,2.001495,11.109070,2.619156;
3,1600,A,252207817872794,0.450623,12.651642,0.18455505;
4,1600,A,252207868226798,-2.164352,13.928436,-4.4224854;


In [15]:
df_csv_phone_accel.set_index(["pessoa_acelelômetro","atividade_acelerômetro","tempo_acelelômetro"],inplace=True)

In [16]:
df_csv_phone_accel.head()

acelelômetro_phone_x  \
pessoa_acelelômetro atividade_acelerômetro tempo_acelelômetro                         
1600                A                      252207666810782                -0.364761   
                                           252207717164786                -0.879730   
                                           252207767518790                 2.001495   
                                           252207817872794                 0.450623   
                                           252207868226798                -2.164352   

                                                               acelelômetro_phone_Y  \
pessoa_acelelômetro atividade_acelerômetro tempo_acelelômetro                         
1600                A                      252207666810782                 8.793503   
                                           252207717164786                 9.768784   
                                           252207767518790                11.109070   
                                           252207817872794                12.651642   
                                           252207868226798                13.928436   

                                                              acelelômetro_phone_Z  
pessoa_acelelômetro atividade_acelerômetro tempo_acelelômetro                       
1600                A                      252207666810782              1.0550842;  
                                           252207717164786              1.0169983;  
                                           252207767518790               2.619156;  
                                           252207817872794             0.18455505;  
                                           252207868226798             -4.4224854;

In [17]:
# Select rows with matching indices (inner join)
common_indices = df_csv_phone_accel.index.intersection(df_csv_phone_gyro.index)
df1_filtered = df_csv_phone_accel.loc[common_indices]
df2_filtered = df_csv_phone_gyro.loc[common_indices]

# Concatenate horizontally
result_concat_phone = pd.concat([df1_filtered, df2_filtered], axis=1)

In [18]:
result_concat_phone.shape

(2787819, 6)

In [19]:
result_concat_phone.head()

acelelômetro_phone_x  acelelômetro_phone_Y  \
1600 A 252207918580802             -4.332779             13.361191   
       252207968934806             -0.319443             13.318359   
       252208019288809              1.566452              9.515274   
       252208069642813             -0.323746              5.262665   
       252208119996817             -1.811676              3.710510   

                       acelelômetro_phone_Z  giroscópio_phone_x  \
1600 A 252207918580802          -0.7188721;           -0.853210   
       252207968934806         -0.23202515;           -0.875137   
       252208019288809         -0.01777649;           -0.720169   
       252208069642813          0.32234192;           -0.571640   
       252208119996817           1.3739319;           -0.380493   

                        giroscópio_phone_Y giroscópio_phone_Z  
1600 A 252207918580802            0.297226         0.8901825;  
       252207968934806            0.015472        0.16223145;  
       252208019288809            0.388489       -0.28401184;  
       252208069642813            1.227402        -0.2416687;  
       252208119996817            1.202835       -0.21313477;

In [20]:
result_concat_phone.tail()

acelelômetro_phone_x  acelelômetro_phone_Y  \
1650 S 357733977635000             -8.509504             -3.520679   
       357734017925000             -8.423284             -4.061953   
       357734058325000             -8.327483             -4.196074   
       357734098619000             -8.080796             -4.418811   
       357734138868000             -8.040081             -4.366121   

                       acelelômetro_phone_Z  giroscópio_phone_x  \
1650 S 357733977635000          -1.8226099;            0.438889   
       357734017925000          -1.7603394;            0.444215   
       357734058325000          -1.5711329;            0.519849   
       357734098619000          -1.4154567;            0.545415   
       357734138868000          -1.4968873;            0.471912   

                        giroscópio_phone_Y giroscópio_phone_Z  
1650 S 357733977635000            0.171508        0.16937704;  
       357734017925000            0.158724        0.19707392;  
       357734058325000            0.037284        0.19600865;  
       357734098619000           -0.086286        0.17896444;  
       357734138868000           -0.201335        0.23222765;

In [21]:
df_csv_watch_gyro = combine_csv_data(csv_watch_gyro_dir)

In [22]:
df_csv_watch_gyro.columns = ['pessoa_gyro','atividade_gyro','tempo_gyro','giroscópio_watch_x','giroscópio_watch_Y','giroscópio_watch_Z']

In [23]:
df_csv_watch_gyro.head()

,pessoa_gyro,atividade_gyro,tempo_gyro,giroscópio_watch_x,giroscópio_watch_Y,giroscópio_watch_Z
0,1600,A,90426757696641,0.314944,-1.022277,-0.3099616;
1,1600,A,90426807196641,0.387382,-0.618541,-0.048971802;
2,1600,A,90426856696641,0.070999,-0.209480,-0.1959783;
3,1600,A,90426906196641,0.037975,0.254976,-0.1565635;
4,1600,A,90426955696641,0.073129,0.719431,-0.0010349044;


In [24]:
df_csv_watch_gyro.shape

(3440342, 6)

In [25]:
df_csv_watch_gyro.set_index(["pessoa_gyro","atividade_gyro","tempo_gyro"],inplace=True)

In [26]:
df_csv_watch_gyro.head()

giroscópio_watch_x  \
pessoa_gyro atividade_gyro tempo_gyro                           
1600        A              90426757696641            0.314944   
                           90426807196641            0.387382   
                           90426856696641            0.070999   
                           90426906196641            0.037975   
                           90426955696641            0.073129   

                                           giroscópio_watch_Y  \
pessoa_gyro atividade_gyro tempo_gyro                           
1600        A              90426757696641           -1.022277   
                           90426807196641           -0.618541   
                           90426856696641           -0.209480   
                           90426906196641            0.254976   
                           90426955696641            0.719431   

                                          giroscópio_watch_Z  
pessoa_gyro atividade_gyro tempo_gyro                         
1600        A              90426757696641        -0.3099616;  
                           90426807196641      -0.048971802;  
                           90426856696641        -0.1959783;  
                           90426906196641        -0.1565635;  
                           90426955696641     -0.0010349044;

In [27]:
df_csv_watch_accel = combine_csv_data(csv_watch_accel_dir)

In [28]:
df_csv_watch_accel.shape

(3777046, 6)

In [29]:
df_csv_watch_accel.columns = ['pessoa_acelelômetro','atividade_acelerômetro','tempo_acelelômetro','acelelômetro_watch_x','acelelômetro_watch_Y','acelelômetro_watch_Z']

In [30]:
df_csv_watch_accel.head()

,pessoa_acelelômetro,atividade_acelerômetro,tempo_acelelômetro,acelelômetro_watch_x,acelelômetro_watch_Y,acelelômetro_watch_Z
0,1600,A,90426708196641,7.091625,-0.591667,8.195502;
1,1600,A,90426757696641,4.972757,-0.158317,6.6967316;
2,1600,A,90426807196641,3.253720,-0.191835,6.107758;
3,1600,A,90426856696641,2.801216,-0.155922,5.997625;
4,1600,A,90426906196641,3.770868,-1.051354,7.731027;


In [31]:
df_csv_watch_accel.set_index(["pessoa_acelelômetro","atividade_acelerômetro","tempo_acelelômetro"],inplace=True)

In [32]:
df_csv_watch_accel.head()

acelelômetro_watch_x  \
pessoa_acelelômetro atividade_acelerômetro tempo_acelelômetro                         
1600                A                      90426708196641                  7.091625   
                                           90426757696641                  4.972757   
                                           90426807196641                  3.253720   
                                           90426856696641                  2.801216   
                                           90426906196641                  3.770868   

                                                               acelelômetro_watch_Y  \
pessoa_acelelômetro atividade_acelerômetro tempo_acelelômetro                         
1600                A                      90426708196641                 -0.591667   
                                           90426757696641                 -0.158317   
                                           90426807196641                 -0.191835   
                                           90426856696641                 -0.155922   
                                           90426906196641                 -1.051354   

                                                              acelelômetro_watch_Z  
pessoa_acelelômetro atividade_acelerômetro tempo_acelelômetro                       
1600                A                      90426708196641                8.195502;  
                                           90426757696641               6.6967316;  
                                           90426807196641                6.107758;  
                                           90426856696641                5.997625;  
                                           90426906196641                7.731027;

In [33]:
# Select rows with matching indices (inner join)
common_indices_watch = df_csv_watch_accel.index.intersection(df_csv_watch_gyro.index)
df1_filtered_watch = df_csv_watch_accel.loc[common_indices_watch]
df2_filtered_watch = df_csv_watch_gyro.loc[common_indices_watch]

# Concatenate horizontally
result_concat_watch = pd.concat([df1_filtered_watch, df2_filtered_watch], axis=1)

In [34]:
result_concat_watch.shape

(3238804, 6)

In [35]:
result_concat_watch.head()

acelelômetro_watch_x  acelelômetro_watch_Y  \
1600 A 90426757696641              4.972757             -0.158317   
       90426807196641              3.253720             -0.191835   
       90426856696641              2.801216             -0.155922   
       90426906196641              3.770868             -1.051354   
       90426955696641              4.661511              0.169689   

                      acelelômetro_watch_Z  giroscópio_watch_x  \
1600 A 90426757696641           6.6967316;            0.314944   
       90426807196641            6.107758;            0.387382   
       90426856696641            5.997625;            0.070999   
       90426906196641            7.731027;            0.037975   
       90426955696641            9.684695;            0.073129   

                       giroscópio_watch_Y giroscópio_watch_Z  
1600 A 90426757696641           -1.022277        -0.3099616;  
       90426807196641           -0.618541      -0.048971802;  
       90426856696641           -0.209480        -0.1959783;  
       90426906196641            0.254976        -0.1565635;  
       90426955696641            0.719431     -0.0010349044;

In [36]:
result_concat_watch.tail()

acelelômetro_watch_x  acelelômetro_watch_Y  \
1650 S 2426184371437252              2.137723             -7.211485   
       2426184421424592              1.486500            -13.302334   
       2426184471411932              4.017171            -13.146711   
       2426184521399272              4.412214            -15.318253   
       2426184571385262              4.426579            -13.012636   

                        acelelômetro_watch_Z  giroscópio_watch_x  \
1650 S 2426184371437252            8.065467;           -2.856065   
       2426184421424592            5.355231;           -2.911458   
       2426184471411932           5.1182046;           -3.015854   
       2426184521399272           3.3704374;           -3.218254   
       2426184571385262           2.5755625;           -2.455525   

                         giroscópio_watch_Y giroscópio_watch_Z  
1650 S 2426184371437252            2.057101        0.85393304;  
       2426184421424592            1.817417         1.2811041;  
       2426184471411932            1.550036         2.0576818;  
       2426184521399272            1.595842         3.0185504;  
       2426184571385262            0.866136         3.3541086;

In [47]:
def custom_concat(df1, df2):
    """
    Concatenates two DataFrames with the same column names, considering specific conditions.

    Args:
        df1 (pd.DataFrame): The first DataFrame.
        df2 (pd.DataFrame): The second DataFrame.

    Returns:
        pd.DataFrame: The concatenated DataFrame.
    """

    result_df = pd.DataFrame()

    # Create groups based on unique combinations of X and Y values
    groups_df1 = df1.groupby(['level_0', 'level_1'])
    groups_df2 = df2.groupby(['level_0', 'level_1'])

    # Iterate over groups from both DataFrames simultaneously
    for (x, y), group1 in groups_df1:
        group2 = groups_df2.get_group((x, y))
        if group2 is not None:  # Check if corresponding group exists in df2
            result_df = pd.concat([result_df, pd.concat([group1, group2], keys=[1, 2], axis=1, ignore_index=True)])

    return result_df

In [38]:
result_concat_watch.reset_index(inplace=True)

In [39]:
result_concat_watch.tail()

,level_0,level_1,level_2,acelelômetro_watch_x,acelelômetro_watch_Y,acelelômetro_watch_Z,giroscópio_watch_x,giroscópio_watch_Y,giroscópio_watch_Z
3238799,1650,S,2426184371437252,2.137723,-7.211485,8.065467;,-2.856065,2.057101,0.85393304;
3238800,1650,S,2426184421424592,1.486500,-13.302334,5.355231;,-2.911458,1.817417,1.2811041;
3238801,1650,S,2426184471411932,4.017171,-13.146711,5.1182046;,-3.015854,1.550036,2.0576818;
3238802,1650,S,2426184521399272,4.412214,-15.318253,3.3704374;,-3.218254,1.595842,3.0185504;
3238803,1650,S,2426184571385262,4.426579,-13.012636,2.5755625;,-2.455525,0.866136,3.3541086;


In [44]:
result_concat_watch.head()

,level_0,level_1,level_2,acelelômetro_watch_x,acelelômetro_watch_Y,acelelômetro_watch_Z,giroscópio_watch_x,giroscópio_watch_Y,giroscópio_watch_Z
0,1600,A,90426757696641,4.972757,-0.158317,6.6967316;,0.314944,-1.022277,-0.3099616;
1,1600,A,90426807196641,3.253720,-0.191835,6.107758;,0.387382,-0.618541,-0.048971802;
2,1600,A,90426856696641,2.801216,-0.155922,5.997625;,0.070999,-0.209480,-0.1959783;
3,1600,A,90426906196641,3.770868,-1.051354,7.731027;,0.037975,0.254976,-0.1565635;
4,1600,A,90426955696641,4.661511,0.169689,9.684695;,0.073129,0.719431,-0.0010349044;


In [74]:
teste = result_concat_phone[(result_concat_phone['level_0'] == 1602) & (result_concat_phone['level_1'] == 'H')]

In [75]:
print(teste)

Empty DataFrame
Columns: [level_0, level_1, level_2, acelelômetro_phone_x, acelelômetro_phone_Y, acelelômetro_phone_Z, giroscópio_phone_x, giroscópio_phone_Y, giroscópio_phone_Z]
Index: []


In [40]:
result_concat_phone.reset_index(inplace=True)

In [45]:
result_concat_phone.head()

,level_0,level_1,level_2,acelelômetro_phone_x,acelelômetro_phone_Y,acelelômetro_phone_Z,giroscópio_phone_x,giroscópio_phone_Y,giroscópio_phone_Z
0,1600,A,252207918580802,-4.332779,13.361191,-0.7188721;,-0.853210,0.297226,0.8901825;
1,1600,A,252207968934806,-0.319443,13.318359,-0.23202515;,-0.875137,0.015472,0.16223145;
2,1600,A,252208019288809,1.566452,9.515274,-0.01777649;,-0.720169,0.388489,-0.28401184;
3,1600,A,252208069642813,-0.323746,5.262665,0.32234192;,-0.571640,1.227402,-0.2416687;
4,1600,A,252208119996817,-1.811676,3.710510,1.3739319;,-0.380493,1.202835,-0.21313477;


In [41]:
result_concat_phone.tail()

,level_0,level_1,level_2,acelelômetro_phone_x,acelelômetro_phone_Y,acelelômetro_phone_Z,giroscópio_phone_x,giroscópio_phone_Y,giroscópio_phone_Z
2787814,1650,S,357733977635000,-8.509504,-3.520679,-1.8226099;,0.438889,0.171508,0.16937704;
2787815,1650,S,357734017925000,-8.423284,-4.061953,-1.7603394;,0.444215,0.158724,0.19707392;
2787816,1650,S,357734058325000,-8.327483,-4.196074,-1.5711329;,0.519849,0.037284,0.19600865;
2787817,1650,S,357734098619000,-8.080796,-4.418811,-1.4154567;,0.545415,-0.086286,0.17896444;
2787818,1650,S,357734138868000,-8.040081,-4.366121,-1.4968873;,0.471912,-0.201335,0.23222765;


In [42]:
result_concat_watch.shape

(3238804, 9)

In [43]:
result_concat_phone.shape

(2787819, 9)

In [48]:
# Concatenate the DataFrames
concatenated_df = custom_concat(result_concat_watch.copy(), result_concat_phone.copy())

KeyError: (1602, 'H')

In [ ]:
result.columns = ['pessoa_x','atividade','tempo','acelerômetro_x', 'acelerômetro_Y', 'acelerômetro_Z', 'giroscópio_x', 'giroscópio_Y', 'giroscópio_Z'] 

In [ ]:
result.head()

In [ ]:
result.atividade.value_counts().sort_index()

In [ ]:
# Define the mapping dictionary
mapping_dict = {'A':0,'B':1,'C':2,'D':3,'E':4,'F':5,'G':6,'H':7,'I':8,'J':9,'K':10,'L':11,'M':12,'O':13,'P':14,'Q':15,'R':16,'S':17}

In [ ]:
result["atividade"] = result["atividade"].replace(to_replace=mapping_dict)

In [ ]:
result.head()

In [ ]:
result.tail()

In [ ]:
cols_to_check = ['acelerômetro_Z','giroscópio_Z']

In [ ]:
result[cols_to_check] = result[cols_to_check].replace({';':''}, regex=True)

In [ ]:
result.head()

In [ ]:
result.to_csv('activity_recognition_data_phone.csv', index=False)